In [1]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.stats import skew, kurtosis

c:\Users\juan_\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.25.905) or chardet (7.6.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
# 1. Dynamically scan dataset path
data_dir = r"C:\Users\juan_\Desktop\cv-medical\COBRE_Dataset"
subject_paths = sorted(glob.glob(os.path.join(data_dir, "sub-*", "anat", "*_T1w.nii.gz")))

feature_database = []
total_subjects = len(subject_paths)

In [5]:
print(f"--- Starting traditional computer vision & radiomics feature extraction ---")
print(f"Total subjects detected: {total_subjects}\n")

# 2. Iterate through each subject with progress logs
for idx, img_path in enumerate(subject_paths, start=1):
    subject_id = os.path.basename(img_path).split('_')[0]
    
    # Print progress log every 10 subjects or on the first/last one
    if idx == 1 or idx % 10 == 0 or idx == total_subjects:
        print(f"Processing subject [{idx}/{total_subjects}] -> {subject_id}...")
    
    try:
        # Load 3D volume
        nii_img = nib.load(img_path)
        volume = nii_img.get_fdata()
        
        # Filter out pure background (zero or near-zero voxels) to avoid biasing statistics
        brain_voxels = volume[volume > (np.max(volume) * 0.05)]
        
        if len(brain_voxels) > 0:
            # Extract traditional statistical and texture-proxy features
            feat_mean = np.mean(brain_voxels)
            feat_std = np.std(brain_voxels)
            feat_median = np.median(brain_voxels)
            feat_skew = skew(brain_voxels)
            feat_kurt = kurtosis(brain_voxels)
            feat_p95 = np.percentile(brain_voxels, 95)
            feat_p05 = np.percentile(brain_voxels, 5)
            
            feature_database.append({
                'participant_id': subject_id,
                'mean_intensity': feat_mean,
                'std_intensity': feat_std,
                'median_intensity': feat_median,
                'skewness': feat_skew,
                'kurtosis': feat_kurt,
                'percentile_95': feat_p95,
                'percentile_05': feat_p05
            })
    except Exception as e:
        print(f"⚠️ Error processing {subject_id}: {e}")

# 3. Convert to DataFrame and display summary
df_features = pd.DataFrame(feature_database)
print(f"\n--- Feature extraction complete! Successfully processed {len(df_features)}/{total_subjects} subjects ---")
print(df_features.head())

--- Starting traditional computer vision & radiomics feature extraction ---
Total subjects detected: 265

Processing subject [1/265] -> sub-10159...
Processing subject [10/265] -> sub-10235...
Processing subject [20/265] -> sub-10316...
Processing subject [30/265] -> sub-10365...
Processing subject [40/265] -> sub-10471...
Processing subject [50/265] -> sub-10530...
Processing subject [60/265] -> sub-10672...
Processing subject [70/265] -> sub-10708...
Processing subject [80/265] -> sub-10871...
Processing subject [90/265] -> sub-10958...
Processing subject [100/265] -> sub-11050...
Processing subject [110/265] -> sub-11088...
Processing subject [120/265] -> sub-11128...
Processing subject [130/265] -> sub-50008...
Processing subject [140/265] -> sub-50025...
Processing subject [150/265] -> sub-50047...
Processing subject [160/265] -> sub-50058...
Processing subject [170/265] -> sub-50076...
Processing subject [180/265] -> sub-60010...
Processing subject [190/265] -> sub-60028...
Proce

## Data Classification

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [7]:
# 1. Ensure we use the pre-extracted feature dataframe and load clinical metadata
# (df_features is already alive in your notebook memory from the previous cell!)
csv_path = r"C:\Users\juan_\Desktop\cv-medical\COBRE_Dataset\participants.tsv"
df_participants = pd.read_csv(csv_path, sep='\t')

In [8]:
# 2. Merge clinical metadata with the extracted image features on 'participant_id'
df_model = pd.merge(df_participants, df_features, on='participant_id', how='inner')

print(f"Total merged samples ready for modeling: {len(df_model)}")

Total merged samples ready for modeling: 265


In [9]:
# 3. Define features (X) and target label (y)
feature_columns = ['mean_intensity', 'std_intensity', 'median_intensity', 'skewness', 'kurtosis', 'percentile_95', 'percentile_05']
X = df_model[feature_columns]
y = df_model['diagnosis']

In [10]:
# Drop any potential NaN values to keep data pristine
valid_idx = X.dropna().index
X = X.loc[valid_idx]
y = y.loc[valid_idx]

In [11]:
# 4. Stratified Train/Test Split (preserving class and demographic proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (212, 7)
Testing set shape: (53, 7)


In [12]:
# 5. Train a robust Random Forest Classifier baseline
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [13]:
# 6. Evaluate model performance
y_pred = rf_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n--- Baseline Model Results ---")
print(f"Accuracy Score: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


--- Baseline Model Results ---
Accuracy Score: 0.4340

Classification Report:
              precision    recall  f1-score   support

        ADHD       0.00      0.00      0.00         8
     BIPOLAR       0.25      0.30      0.27        10
     CONTROL       0.50      0.72      0.59        25
        SCHZ       0.50      0.20      0.29        10

    accuracy                           0.43        53
   macro avg       0.31      0.30      0.29        53
weighted avg       0.38      0.43      0.38        53

